In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, tf_to_int, tf_to_daily, tf_to_hourly, print_baseline_energy_data, print_opt_energy_data
from modules.visualisations import plot_profile_by_category,plot_distribution_comparison,plot_sorted_mps_comparison

from plotly.io import to_html
from IPython.display import display, HTML

from modules.optimization_algorithms import NashProductOptAlgo, DefaultOptAlgo, EqualWaterfillingOptAlgo, NoOptAlgo
from modules.optimization_algorithms.OptAlgo import OptAlgo
from datetime import datetime, UTC

from pathlib import Path
import json

from modules.pf_calculations import plot_stacked_gain_loss_sortable


In [ ]:
# 
# LOAD ENPARTO LOGO
# 
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

- from the perspective of a GEN MP: 
    - how much is produced
    - how much is distributed in the REC
    - how much is now distributed in the REC
    - how much is now distributed in X
    - how much is distributed IN TOTAL  / change of total surplus

- from the perspective of a CONS MP: 
    - how much is consumed
    - how much is covered by the REC
    - how much is now covered by the REC
    - how much is now covered by the REC
    - how much is covered IN TOTAL  / change of total Comm Cov

- from the perspective of a single REC:
    - sums of all energy flow values
    - sums of all energy flow INTERNAL values
    - sums of all energy flow to X values

## Params

In [ ]:
# params -> 1, 13, Jan 7 - 28, hourly, max, max, max, max, DefauktOptAlgo -> works super good

In [ ]:
# # PARMS
# changeable
org_A_id = 1
org_B_id = 35

consumer_org_ids = [org_A_id]  # [1, 2]
prosumer_org_ids = [org_B_id]  # [13, 17]

start_time = datetime(2025, 1, 7)
end_time = datetime(2025, 1, 28)

tf_to_use = "hourly" # "15min" # "daily" # "hourly"

sim_eeg_hourly_pf_mode = "max"
sim_eeg_daily_pf_mode = "max"

org_algo_mapping = {
    org_A_id: {
        "opt_algo": DefaultOptAlgo,
        "share_ratio": 1,
        "hourly_pf_mode": "max",
        "daily_pf_mode": "max",
    },
    org_B_id: {
        "opt_algo": DefaultOptAlgo,
        "share_ratio": 1,
        "hourly_pf_mode": "max",
        "daily_pf_mode": "max",
    },
    "x": {
        "opt_algo": NoOptAlgo,
        "share_ratio": 1,
    },
}

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

## load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename_template = config_dict["SINGLE_MPS_OF_EEG"]

In [ ]:
raw_eeg = []

for eeg_type, eeg_ids_of_type in [
    ("c", consumer_org_ids),
    ("p", prosumer_org_ids),
]:
    for act_org_id in eeg_ids_of_type:
        act_filepath_to_load = (
            f"{path_to_local_data}{input_filename_template.format(org_id=act_org_id)}"
        )
        act_df = pd.read_csv(act_filepath_to_load)
        act_df["eeg_type"] = eeg_type
        raw_eeg.append(act_df)

# row-wise append into a single DataFrame
raw_eeg_df = pd.concat(raw_eeg, ignore_index=True)

raw_eeg_df['time'] = pd.to_datetime(raw_eeg_df['time'], utc=True)

eeg_selected_time_horizon = raw_eeg_df[
    (raw_eeg_df["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (raw_eeg_df["time"] < pd.Timestamp(end_time, tz='UTC'))
]
del raw_eeg_df



print(f"{eeg_selected_time_horizon.dtypes}")
print(f"len: {len(eeg_selected_time_horizon)}")
eeg_selected_time_horizon.head()

In [ ]:
eeg_selected_feat = eeg_selected_time_horizon[["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del eeg_selected_time_horizon

# cons_gen: Consumed Generation, how much of the generated electricity was consumed within the EEG
eeg_selected_feat.sum(numeric_only=True)

# PRINT DF INFO
# 
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop(["organization_id", "metering_point_id"], axis=1).describe())
eeg_selected_feat.head()

#
# FEATURE ENGINEERIN
#

eeg_selected_feat["cons_gen"] = eeg_selected_feat["wt_meas_gen"] - eeg_selected_feat["wt_surp_gen"]

In [ ]:
#
# DETECT NON-PV GENERATION
# 

# mp_id, where energy_direction = "G", and q75 of wt_meas_gen between 11 and 3 is greater than 1.

## Apply TF Opt on source EEGs

In [ ]:
work = eeg_selected_feat.drop_duplicates().copy()
del eeg_selected_feat

# result container (separate from config!)
simulated_energy_data = {}

run_timestamp = datetime.now(UTC).isoformat()

for act_org_id, act_algo_mapping in org_algo_mapping.items():  # work["organization_id"].unique():
    if act_org_id == "x":
        continue
    print(f"start opt {act_org_id}:")

    algo_cls = act_algo_mapping["opt_algo"]
    act_algo: OptAlgo = algo_cls()

    act_work = work[work["organization_id"] == act_org_id].copy()

    act_tf_schedule = act_algo.calculate_pfs(act_work)
    act_tf_int_schedule = tf_to_int(act_tf_schedule)
    act_tf_hourly_schedule = tf_to_hourly(act_tf_int_schedule, mode=act_algo_mapping["hourly_pf_mode"])
    act_tf_daily_schedule = tf_to_daily(act_tf_hourly_schedule, mode=act_algo_mapping["daily_pf_mode"])
        
    if tf_to_use == "hourly":
        print("applying hourly pfs")
        act_tf_to_use = act_tf_hourly_schedule
    elif tf_to_use == "daily":
        print("applying daily pfs")        
        act_tf_to_use = act_tf_daily_schedule
    else:
        print("applying default 15min pfs")
        act_tf_to_use = act_tf_int_schedule

    act_sim_ed = apply_pf_schedule_to_mps(act_work, act_tf_to_use)

    simulated_energy_data[act_org_id] = {
        "raw_tf_schedule": act_tf_schedule,
        "used_tf_schedule": act_tf_to_use,
        "simulated_energy_data": act_sim_ed,
        "algo_name": algo_cls.__name__,
        "algo_params": vars(act_algo),  # assumes params stored on self
        "created_at": run_timestamp,
    }
    print(f"end opt {act_org_id}:")

## Calculate new combined EEG

In [ ]:
temp_work = pd.concat([simulated_energy_data[org_A_id]["simulated_energy_data"], simulated_energy_data[org_B_id]["simulated_energy_data"]]).copy()

temp_work["rest_wt_meas_gen"] = temp_work["wt_meas_gen"] * (1 - temp_work["pf"]) 
temp_work["rest_wt_meas_cons"] = temp_work["wt_meas_cons"] * (1 - temp_work["pf"]) 

# extract org_id -> share_ratio

share_ratio_map = {org_id: cfg["share_ratio"] for org_id, cfg in org_algo_mapping.items()}
temp_work["transfer_ratio"] = temp_work["organization_id"].map(share_ratio_map)
temp_work["transfer_wt_meas_cons"] = temp_work["rest_wt_meas_cons"] * temp_work["transfer_ratio"]
temp_work["transfer_wt_meas_gen"] = temp_work["rest_wt_meas_gen"] * temp_work["transfer_ratio"]

In [ ]:
# temp_work[temp_work["pf"]!=1][["organization_id", "wt_meas_gen", "opt_wt_meas_gen", "rest_wt_meas_gen", "transfer_wt_meas_gen", "transfer_ratio"]]

temp_work[["organization_id", "wt_meas_gen", "opt_wt_meas_gen", "rest_wt_meas_gen", "transfer_wt_meas_gen", "transfer_ratio"]]


temp_sim_eeg_combination = temp_work[["organization_id", "metering_point_id", "energy_direction", "time", "transfer_wt_meas_gen", "transfer_wt_meas_cons"]].rename(columns={"transfer_wt_meas_gen":"wt_meas_gen", "transfer_wt_meas_cons":"wt_meas_cons"})

new_calced_agg_on_time = temp_sim_eeg_combination.groupby(by="time").sum().reset_index()[["time", "wt_meas_cons", "wt_meas_gen"]].rename(columns={"wt_meas_cons":"sum_wt_meas_cons", "wt_meas_gen":"sum_wt_meas_gen"})
temp_sim_eeg_combination = temp_sim_eeg_combination.merge(new_calced_agg_on_time, on=["time"], how="left")


temp_sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio"] = (temp_sim_eeg_combination["sum_wt_meas_gen"] / temp_sim_eeg_combination["sum_wt_meas_cons"]).replace(np.nan, 1)
temp_sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio_clipped"] = temp_sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio"].clip(upper=1) 
temp_sim_eeg_combination["wt_meas_surp_to_wt_meas_gen_ratio_clipped"] = ((temp_sim_eeg_combination["sum_wt_meas_gen"] - temp_sim_eeg_combination["sum_wt_meas_cons"])/temp_sim_eeg_combination["sum_wt_meas_gen"]).clip(lower=0) 

# applying new rations to calculate new comm_cov, comm_pot and wt_meas_surp after pf schedule application
temp_sim_eeg_combination["comm_cov"] = temp_sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio_clipped"] * temp_sim_eeg_combination["wt_meas_cons"]
temp_sim_eeg_combination["comm_pot"] = temp_sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio"] * temp_sim_eeg_combination["wt_meas_cons"]
temp_sim_eeg_combination["wt_surp_gen"] = temp_sim_eeg_combination["wt_meas_surp_to_wt_meas_gen_ratio_clipped"] * temp_sim_eeg_combination["wt_meas_gen"]
temp_sim_eeg_combination["cons_gen"] = temp_sim_eeg_combination["wt_meas_gen"] - temp_sim_eeg_combination["wt_surp_gen"]

temp_sim_eeg_combination["source_organization_id"]=temp_sim_eeg_combination["organization_id"]
temp_sim_eeg_combination["organization_id"]="x"

sim_eeg_combination = temp_sim_eeg_combination.drop(columns=["sum_wt_meas_cons", "sum_wt_meas_gen", "wt_meas_gen_to_wt_meas_cons_ratio", "wt_meas_gen_to_wt_meas_cons_ratio_clipped", "wt_meas_surp_to_wt_meas_gen_ratio_clipped"])
sim_eeg_combination.head()

## Apply TF Opt on new combined EEG

In [ ]:
act_org_id = "x"
print(f"start opt {act_org_id}:")

algo_cls = org_algo_mapping[act_org_id]["opt_algo"]
act_algo = algo_cls()

act_work = sim_eeg_combination[
    sim_eeg_combination["organization_id"] == act_org_id
].copy()

act_tf_schedule = act_algo.calculate_pfs(act_work)
act_tf_int_schedule = tf_to_int(act_tf_schedule)
act_tf_hourly_schedule = tf_to_hourly(act_tf_int_schedule, mode=sim_eeg_hourly_pf_mode)
act_tf_daily_schedule = tf_to_daily(act_tf_hourly_schedule, mode=sim_eeg_daily_pf_mode)

if tf_to_use == "hourly":
    print("applying hourly pfs")
    act_tf_to_use = act_tf_hourly_schedule
elif tf_to_use == "daily":
    print("applying daily pfs")
    act_tf_to_use = act_tf_daily_schedule
else:
    print("applying default 15min pfs")
    act_tf_to_use = act_tf_int_schedule

act_sim_ed = apply_pf_schedule_to_mps(act_work, act_tf_to_use)

simulated_energy_data[act_org_id] = {
    "raw_tf_schedule": act_tf_schedule,
    "used_tf_schedule": act_tf_to_use,
    "simulated_energy_data": act_sim_ed,
    "algo_name": algo_cls.__name__,
    "algo_params": vars(act_algo),  # assumes params stored on self
    "created_at": run_timestamp,
}
print(f"end opt {act_org_id}:")

In [ ]:
for act_org_id, act_sim_ed in simulated_energy_data.items():
    print(f"{'='*20}\n{act_org_id}")
    if act_org_id != "x":
        print_opt_energy_data(act_sim_ed["simulated_energy_data"])
    else:
        print_baseline_energy_data(act_sim_ed["simulated_energy_data"])
    print(f"{'='*20}\n")

## Evaluate overall energy flow Balance

In [ ]:
source_eeg_suffix = "_s"
combined_eeg_suffix = "_c"

comb_eegs_a = simulated_energy_data[org_A_id]["simulated_energy_data"].merge(
    simulated_energy_data["x"]["simulated_energy_data"],
    on=["time", "metering_point_id", "energy_direction"],
    suffixes=(source_eeg_suffix, combined_eeg_suffix),
)

comb_eegs_b = simulated_energy_data[org_B_id]["simulated_energy_data"].merge(
    simulated_energy_data["x"]["simulated_energy_data"],
    on=["time", "metering_point_id", "energy_direction"],
    suffixes=(source_eeg_suffix, combined_eeg_suffix),
)

comb_eegs = pd.concat([comb_eegs_a, comb_eegs_b], ignore_index=True)

comb_eegs = comb_eegs.drop(
    columns=[c for c in comb_eegs.columns if c.startswith("sum_")]
)

comb_eegs[f"pf{source_eeg_suffix}_to{combined_eeg_suffix}"] = comb_eegs[f"pf{combined_eeg_suffix}"]
comb_eegs[f"pf{combined_eeg_suffix}"] = comb_eegs[f"pf{source_eeg_suffix}_to{combined_eeg_suffix}"] * comb_eegs[f"pf{source_eeg_suffix}"]

comb_eegs["comm_cov_f"] = comb_eegs["opt_comm_cov_s"] + comb_eegs["comm_cov_c"]
comb_eegs["opt_comm_cov_f"] = comb_eegs["opt_comm_cov_s"] + comb_eegs["opt_comm_cov_c"]
comb_eegs["comm_cov_f_diff"] = comb_eegs["comm_cov_f"] - comb_eegs["comm_cov_s"]

comb_eegs["cons_gen_f"] = comb_eegs["opt_cons_gen_s"] + comb_eegs["cons_gen_c"]
comb_eegs["opt_cons_gen_f"] = comb_eegs["opt_cons_gen_s"] + comb_eegs["opt_cons_gen_c"]
comb_eegs["cons_gen_f_diff"] = comb_eegs["cons_gen_f"] - comb_eegs["cons_gen_s"]



### Consumers

In [ ]:
agg_map_consumers = {
    "organization_id_s": "first",
    f"wt_meas_cons{source_eeg_suffix}": "sum",
    f"comm_cov{source_eeg_suffix}": "sum",
    f"opt_wt_meas_cons{source_eeg_suffix}": "sum",
    f"opt_comm_cov{source_eeg_suffix}": "sum",
    f"wt_meas_cons{combined_eeg_suffix}": "sum",
    f"comm_cov{combined_eeg_suffix}": "sum",
    f"opt_wt_meas_cons{combined_eeg_suffix}": "sum",
    f"opt_comm_cov{combined_eeg_suffix}": "sum",
    "comm_cov_f":"sum",
    "opt_comm_cov_f":"sum",
    "comm_cov_f_diff":"sum"
}

summed_consumers = (
    comb_eegs[comb_eegs["energy_direction"] == "C"]
    .groupby("metering_point_id")
    .agg(agg_map_consumers)
)



In [ ]:
plot_stacked_gain_loss_sortable(summed_consumers[summed_consumers["organization_id_s"] == org_A_id], "comm_cov_s", "comm_cov_f")

In [ ]:
plot_stacked_gain_loss_sortable(summed_consumers[summed_consumers["organization_id_s"] == org_B_id], "comm_cov_s", "comm_cov_f")

### Generators

In [ ]:
agg_map_generators = {
    "organization_id_s": "first",
    f"wt_meas_gen{source_eeg_suffix}": "sum",
    f"cons_gen{source_eeg_suffix}": "sum",
    f"opt_wt_meas_gen{source_eeg_suffix}": "sum",
    f"opt_cons_gen{source_eeg_suffix}": "sum",
    f"wt_meas_gen{combined_eeg_suffix}": "sum",
    f"cons_gen{combined_eeg_suffix}": "sum",
    f"opt_wt_meas_gen{combined_eeg_suffix}": "sum",
    f"opt_cons_gen{combined_eeg_suffix}": "sum",
    "cons_gen_f":"sum",
    "opt_cons_gen_f":"sum",
    "cons_gen_f_diff":"sum"
}

summed_generators = (
    comb_eegs[comb_eegs["energy_direction"] == "G"]
    .groupby("metering_point_id")
    .agg(agg_map_generators)
)


In [ ]:
plot_stacked_gain_loss_sortable(summed_generators[summed_generators["organization_id_s"] == org_A_id], "cons_gen_s", "cons_gen_f")

In [ ]:
plot_stacked_gain_loss_sortable(summed_generators[summed_generators["organization_id_s"] == org_B_id], "cons_gen_s", "cons_gen_f")

In [ ]:
summed_generators[summed_generators.index == 7791]

In [ ]:
summed_generators_by_month = (
    comb_eegs[comb_eegs["energy_direction"] == "G"]
    .groupby(["metering_point_id", comb_eegs["time"].dt.month])
    .agg(agg_map_generators)
)

In [ ]:
summed_generators_by_month.groupby(["organization_id_s", "time"])[["cons_gen_f_diff"]].sum().reset_index()

In [ ]:
summed_consumers.groupby(["organization_id_s", ])["comm_cov_f_diff"].sum()

In [ ]:
sns.lineplot(comb_eegs[(comb_eegs["organization_id_s"] == 1) & (comb_eegs["energy_direction"]=="C")].groupby(comb_eegs["time"].dt.hour).mean(numeric_only=True)[["comm_cov_c"]])

In [ ]:
sns.lineplot(comb_eegs[(comb_eegs["organization_id_s"] == 1) & (comb_eegs["energy_direction"]=="C")].groupby(comb_eegs["time"].dt.hour).mean(numeric_only=True)[["comm_cov_s"]])

In [ ]:
temp = comb_eegs
energy_cols = ["wt_meas_gen_s", "wt_meas_cons_s", "wt_surp_gen_s", "comm_cov_s", "comm_cov_c"]

In [ ]:
import plotly.graph_objects as go
import plotly.express as px


def plot_aggregate_daily_profile(
    df,
    energy_cols,  # list of energy columns
    agg_func_str="mean",
    extra_col=None,  # e.g. temperature
    logo=None,
):
    if agg_func_str not in ["mean", "median", "sum", "min", "max", "std"]:
        raise ValueError(f"Unsupported aggregation function: {agg_func_str}")

    if isinstance(energy_cols, str):
        energy_cols = [energy_cols]

    for col in energy_cols:
        if col not in df.columns:
            raise ValueError(f"'{col}' is not a column in the DataFrame!")

    if extra_col and extra_col not in df.columns:
        raise ValueError(f"'{extra_col}' is not a column in the DataFrame!")

    # create time of day
    df = df.copy()
    df["daytime"] = df.time.dt.strftime("%H:%M")

    # aggregate over the whole period → 24h profile only
    agg_df = (
        df.groupby("daytime")[energy_cols]
        .agg(agg_func_str)
        .reset_index()
        .sort_values("daytime")
    )

    colors = px.colors.qualitative.Plotly
    fig = go.Figure()

    # energy traces
    for i, col in enumerate(energy_cols):
        fig.add_trace(
            go.Scatter(
                x=agg_df["daytime"],
                y=agg_df[col],
                mode="lines",
                name=f"{col} ({agg_func_str})",
                line=dict(color=colors[i % len(colors)]),
                yaxis="y",
            )
        )

    # optional: additional column on the secondary axis
    if extra_col:
        extra_df = (
            df.groupby("daytime")[extra_col]
            .agg(agg_func_str)
            .reset_index()
            .sort_values("daytime")
        )

        fig.add_trace(
            go.Scatter(
                x=extra_df["daytime"],
                y=extra_df[extra_col],
                mode="lines",
                name=f"{extra_col} ({agg_func_str})",
                line=dict(color="black", dash="dot"),
                yaxis="y2",
            )
        )

        fig.update_layout(
            yaxis2=dict(
                title=extra_col,
                overlaying="y",
                side="right",
                showgrid=False,
                visible=True,
            )
        )

    if logo is not None:
        fig.add_layout_image(logo)

    # layout
    fig.update_layout(
        title=f"Aggregated daily load profile ({agg_func_str})",
        xaxis=dict(
            title="Time of day",
            tickangle=45,
            automargin=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(title=f"Energy ({agg_func_str})"),
        legend=dict(x=0.5, y=1.15, orientation="h", xanchor="center"),
        margin=dict(b=80, t=80, l=60, r=80),
        height=600,
    )

    fig.show()


In [ ]:
plot_profile_by_category(temp[temp["organization_id_s"] == 35], energy_col_name="wt_meas_gen_s", agg_func_str='mean', hue_col="metering_point_id", logo=logo)

In [ ]:
plot_aggregate_daily_profile(temp[temp["organization_id_s"] == 35], energy_cols=energy_cols, agg_func_str='mean', logo=logo)

In [ ]:
temp = comb_eegs
energy_cols = ["wt_meas_gen_s", "wt_meas_cons_s", "wt_surp_gen_s", "cons_gen_s", "cons_gen_c"]
plot_aggregate_daily_profile(temp[temp["organization_id_s"] == 35], energy_cols=energy_cols, agg_func_str='mean', logo=logo)

# 2 RECs

# 3 scenarios
- both surplus: do nothing
- both under-coverage: do nothing
- 1 surplus, 1 under-coverage
